## Rare vs Frequent Labels in Categorical Features

Categorical variables often have **labels with very different frequencies**. Understanding their impact is crucial in feature engineering and modeling.

### 1. Frequent Labels

**Definition:**  
Labels that appear often in the dataset.

**Impact on Models:**

- Well-represented in training data → models can **learn patterns reliably**.
- Contribute to **stable predictions**.
- Can dominate model behavior if not balanced with other features.

**Example:**  
In a `Product` column: `'A'` appears 5000 times → model sees enough examples to learn its relationship with the target.

### 2. Rare Labels

**Definition:**  
Labels that appear infrequently, sometimes only a few times.

**Impact on Models:**

- Sparse representation → model may **overfit** to these rare labels.
- Can introduce **noise** if the label’s target values are inconsistent.
- Can increase **dimensionality** when one-hot encoding, leading to memory issues.
- May cause instability in statistical encoding methods like target encoding.

**Example:**  
In a `Product` column: `'Z'` appears only 2 times → model may learn a spurious pattern.

### 3. Industry Handling

- **Group rare labels** into `"Other"` to reduce dimensionality and noise.
- **Frequency encoding**: replace rare labels with their counts or frequencies.
- **Target encoding with smoothing**: reduces overfitting by weighting rare labels toward the global mean.
- Rare labels can be **flagged** for monitoring or special handling in business rules.


We have already covered **grouping rare labels** and **frequency encoding** as ways to handle high-cardinality categorical features. Another powerful method is **target encoding with smoothing**, which is especially useful for supervised tasks.

**Definition:**  
Target encoding replaces each category with a statistic of the target variable (commonly the mean) for that category. **Smoothing** is added to reduce overfitting for **rare labels** by combining the category mean with the **global mean**.

**Formula (simplified):**

$$Encoded_{value} = \frac{\text{mean target category} \times n_{category} + global_{mean} \times \alpha}{n_{category} + \alpha}$$

- $n_{category}$ → number of occurrences of the category
- $\alpha$ → smoothing parameter (higher → more weight to global mean)
- Rare categories get values closer to the **global mean**, reducing overfitting.


In [1]:
import pandas as pd
import numpy as np

np.random.seed(0)

# Simulate a large dataset with rare lables
products = [f'Product_Category_{i}' for i in range(1, 10001)]

weights = np.random.zipf(a=2, size=len(products))
weights = weights / weights.sum()
order_products = np.random.choice(products, size=50000, replace=True, p=weights)

revenue = np.random.normal(loc=100, scale=20, size=50000)

df = pd.DataFrame({'Product': order_products, 'Revenue': revenue})
df

,Product,Revenue
0,Product_Category_7392,82.866855
1,Product_Category_936,105.461523
2,Product_Category_6198,114.456431
3,Product_Category_1149,85.908078
4,Product_Category_1149,83.167623
...,...,...
49995,Product_Category_9844,83.386339
49996,Product_Category_9324,114.412456
49997,Product_Category_6218,102.884015
49998,Product_Category_6198,78.202766


In [2]:
global_mean = df['Revenue'].mean()
category_counts = df['Product'].value_counts()
category_means = df.groupby('Product')['Revenue'].mean()

print("category counts\n", category_counts)
print("\n")
print("category mean\n", category_means)

category counts
 Product
Product_Category_6198    12652
Product_Category_1149     9360
Product_Category_402      2448
Product_Category_6482     2383
Product_Category_6848     1190
                         ...  
Product_Category_4451        1
Product_Category_7522        1
Product_Category_9308        1
Product_Category_3965        1
Product_Category_6218        1
Name: count, Length: 5479, dtype: int64


category mean
 Product
Product_Category_10      104.591078
Product_Category_1001     85.568857
Product_Category_1005     80.548013
Product_Category_1007     65.739835
Product_Category_1008     84.960467
                            ...    
Product_Category_9993     91.311181
Product_Category_9995     65.774469
Product_Category_9997    106.089853
Product_Category_9998    121.976167
Product_Category_9999     92.513943
Name: Revenue, Length: 5479, dtype: float64


In [3]:
alpha = 2  # smoothing factor

def target_encode_smooth(x):
    n = category_counts[x]
    mean = category_means[x]
    return (mean * n + global_mean * alpha) / (n + alpha)

df['Product_encoded'] = df['Product'].apply(target_encode_smooth)
df

,Product,Revenue,Product_encoded
0,Product_Category_7392,82.866855,96.753096
1,Product_Category_936,105.461523,96.101577
2,Product_Category_6198,114.456431,100.153566
3,Product_Category_1149,85.908078,99.925969
4,Product_Category_1149,83.167623,99.925969
...,...,...,...
49995,Product_Category_9844,83.386339,95.162215
49996,Product_Category_9324,114.412456,104.909899
49997,Product_Category_6218,102.884015,101.067085
49998,Product_Category_6198,78.202766,100.153566
